# Optimización de hiperparámetros — modelos ganadores por perfil × horizonte

Este notebook reutiliza EXACTAMENTE la misma infraestructura del notebook
`Modelado_segmentado_comparacion_modelos_3_6_meses` (carga de datos, matrices,
funciones de features, métricas, baseline y validación temporal) para
garantizar que la búsqueda de hiperparámetros se evalúa con el mismo criterio
que se usó para elegir el algoritmo ganador de cada perfil × horizonte.

Reglas que se respetan sin excepción:

- La búsqueda de hiperparámetros optimiza sobre las mismas ventanas de
  validación (`CORTES_VALIDACION` = 2024-07 y 2025-01) que ya se usaban.
- El **backtest final (2025-07-01) nunca se toca aquí** — sigue siendo el
  juez independiente, igual que antes.
- No se usa *early stopping* con las ventanas de validación como `eval_set`:
  eso filtraría información de validación hacia el entrenamiento. En su
  lugar, `n_estimators`/`iterations` es simplemente otro hiperparámetro más
  que Optuna ajusta, igual que se hacía manualmente en el notebook original.
- Para cada intento (*trial*) se recalcula el blend óptimo con la misma
  grilla `GRID_ALPHA`, así que el número que Optuna minimiza es el mismo
  `WAPE_blend_validacion_pct` que ya conoces — los resultados son
  directamente comparables contra `seleccion_modelo_por_perfil_horizonte.csv`.
- Los estudios de Optuna se guardan en un archivo SQLite
  (`optuna_estudios.sqlite3`), así que si el kernel se cae o cierras el
  computador a mitad de la búsqueda, puedes volver a correr la celda del
  bucle principal y continúa donde quedó, sin perder los intentos ya hechos.


## Dependencias

Si alguna librería de modelos no está instalada:

```python
%pip install -U lightgbm xgboost catboost scikit-learn pyarrow joblib
```

Después reinicia el kernel y ejecuta nuevamente desde el inicio.

In [ ]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

from pathlib import Path
import os
import gc
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import joblib

try:
    from lightgbm import (
        LGBMRegressor,
        LGBMClassifier,
    )
except ImportError as e:
    raise ImportError(
        "Falta LightGBM. Ejecuta: %pip install -U lightgbm"
    ) from e

try:
    from xgboost import (
        XGBRegressor,
        XGBClassifier,
    )
except ImportError as e:
    raise ImportError(
        "Falta XGBoost. Ejecuta: %pip install -U xgboost"
    ) from e

try:
    from catboost import (
        CatBoostRegressor,
        CatBoostClassifier,
    )
except ImportError as e:
    raise ImportError(
        "Falta CatBoost. Ejecuta: %pip install -U catboost"
    ) from e

warnings.filterwarnings("ignore")

BASE_DIR = Path(
    os.environ.get("EBSA_DATOS", r"C:\Users\Home\Documents\Datos_Ebsa")
)

PREPROC_DIR = (
    BASE_DIR
    / "03_serie_modelado"
)

RUTA_ENTRADA = (
    PREPROC_DIR
    / "serie_mensual_modelado_preprocesada.parquet"
)

SALIDA_DIR = (
    BASE_DIR
    / "04_pronostico" / "modelo_final"
)

SALIDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUTA_PERFILES_FINAL = (
    SALIDA_DIR
    / "perfiles_consumidores_corte_final.parquet"
)

RUTA_GRANDES_FINAL = (
    SALIDA_DIR
    / "grandes_consumidores_corte_final.parquet"
)

RUTA_AUDITORIA_UMBRALES = (
    SALIDA_DIR
    / "auditoria_umbrales_grandes_consumidores.csv"
)

RUTA_COMPARACION_VALIDACION = (
    SALIDA_DIR
    / "comparacion_modelos_validacion.csv"
)

RUTA_SELECCION_MODELOS = (
    SALIDA_DIR
    / "seleccion_modelo_por_perfil_horizonte.csv"
)

RUTA_TOTALES_VALIDACION = (
    SALIDA_DIR
    / "real_vs_modelos_validacion_totales.csv"
)

RUTA_METRICAS_BACKTEST = (
    SALIDA_DIR
    / "metricas_sistema_ganador_backtest.csv"
)

RUTA_METRICAS_PERFIL = (
    SALIDA_DIR
    / "metricas_sistema_por_perfil_horizonte.csv"
)

RUTA_METRICAS_REGIMEN = (
    SALIDA_DIR
    / "metricas_sistema_por_regimen_horizonte.csv"
)

RUTA_REAL_VS_PRED = (
    SALIDA_DIR
    / "real_vs_pronosticado_sistema_ganador_backtest.parquet"
)

RUTA_MODELOS = (
    SALIDA_DIR
    / "modelos_ganadores_segmentados.joblib"
)

RUTA_PRED_3M = (
    SALIDA_DIR
    / "predicciones_segmentadas_ganadoras_3_meses.parquet"
)

RUTA_PRED_6M = (
    SALIDA_DIR
    / "predicciones_segmentadas_ganadoras_6_meses.parquet"
)

print("Entrada :", RUTA_ENTRADA)
print("Salidas :", SALIDA_DIR)

In [ ]:
# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

SEED = 42

HORIZONTES = [
    1, 2, 3, 4, 5, 6
]

MODELOS_CANDIDATOS = [
    "LightGBM",
    "XGBoost",
    "CatBoost",
]

# ------------------------------------------------------------
# Segmentación
# ------------------------------------------------------------

MIN_MESES_VALIDOS_12 = 6

UMBRAL_MUY_BAJO_KWH = 10.0
UMBRAL_ALTO_KWH = 500.0
UMBRAL_GRANDE_KWH = 5_000.0

PCT_CEROS_INTERMITENTE = 0.50

PERFILES = [
    "P0_INTERMITENTE",
    "P1_REGULAR",
    "P2_ALTO",
    "P3_GRANDE",
    "P4_INSUFICIENTE",
]

PERFILES_MODELADOS = [
    "P0_INTERMITENTE",
    "P1_REGULAR",
    "P2_ALTO",
    "P3_GRANDE",
]

# ------------------------------------------------------------
# Muestreo de entrenamiento
# P2/P3 usan todos los candidatos disponibles.
# ------------------------------------------------------------

MAX_MUESTRA_POR_ORIGEN = {
    "P0_INTERMITENTE": 40_000,
    "P1_REGULAR": 60_000,
    "P2_ALTO": None,
    "P3_GRANDE": None,
}

# ------------------------------------------------------------
# Ventanas temporales
# ------------------------------------------------------------

PRIMER_ORIGEN_TRAIN = pd.Timestamp(
    "2023-01-01"
)

# Los tres modelos se comparan entrenando con targets
# conocidos hasta julio de 2024.
MAX_TARGET_TRAIN_COMPARACION = pd.Timestamp(
    "2024-07-01"
)

# Ventanas utilizadas para seleccionar algoritmo y alpha.
CORTES_VALIDACION = [
    pd.Timestamp("2024-07-01"),
    pd.Timestamp("2025-01-01"),
]

# Backtest completamente posterior.
CORTE_BACKTEST_FINAL = pd.Timestamp(
    "2025-07-01"
)

# Búsqueda de peso ML vs baseline.
GRID_ALPHA = np.round(
    np.linspace(
        0.0,
        1.0,
        21,
    ),
    2,
)

print("MODELOS:", MODELOS_CANDIDATOS)

print("\nPERFILES")
print("-" * 60)

print(
    f"P0: mediana <= {UMBRAL_MUY_BAJO_KWH:,.0f} kWh "
    f"o ceros >= {PCT_CEROS_INTERMITENTE:.0%}"
)

print(
    f"P1: > {UMBRAL_MUY_BAJO_KWH:,.0f} "
    f"y < {UMBRAL_ALTO_KWH:,.0f} kWh"
)

print(
    f"P2: {UMBRAL_ALTO_KWH:,.0f} "
    f"a < {UMBRAL_GRANDE_KWH:,.0f} kWh"
)

print(
    f"P3: >= {UMBRAL_GRANDE_KWH:,.0f} kWh "
    "de mediana histórica 12m"
)

print(
    f"P4: < {MIN_MESES_VALIDOS_12} meses válidos"
)

In [ ]:
# ============================================================
# 3. CARGAR Y VALIDAR ARCHIVO PREPROCESADO
# ============================================================

if not RUTA_ENTRADA.exists():
    raise FileNotFoundError(
        f"No existe el archivo:\n{RUTA_ENTRADA}"
    )

serie = pd.read_parquet(
    RUTA_ENTRADA,
    engine="pyarrow",
)

obligatorias = [
    "NIU",
    "periodo",
    "consumo_kwh_mensual",
]

faltantes = [
    c
    for c in obligatorias
    if c not in serie.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas obligatorias: {faltantes}"
    )

serie["NIU"] = (
    serie["NIU"]
    .astype("string")
    .str.strip()
)

serie["periodo"] = pd.to_datetime(
    serie["periodo"],
    errors="coerce",
)

serie["consumo_kwh_mensual"] = pd.to_numeric(
    serie["consumo_kwh_mensual"],
    errors="coerce",
).astype("float32")

duplicados = int(
    serie.duplicated(
        subset=[
            "NIU",
            "periodo",
        ]
    ).sum()
)

negativos = int(
    serie[
        "consumo_kwh_mensual"
    ]
    .lt(0)
    .sum()
)

print("VALIDACIÓN DE ENTRADA")
print("-" * 60)
print(f"Filas                  : {len(serie):,}")
print(f"NIU únicos             : {serie['NIU'].nunique():,}")
print(
    f"Periodo                : "
    f"{serie['periodo'].min():%Y-%m} "
    f"→ {serie['periodo'].max():%Y-%m}"
)
print(f"Duplicados NIU-periodo : {duplicados:,}")
print(
    f"Consumos nulos         : "
    f"{serie['consumo_kwh_mensual'].isna().sum():,}"
)
print(
    f"Consumos cero          : "
    f"{serie['consumo_kwh_mensual'].eq(0).sum():,}"
)
print(f"Consumos negativos     : {negativos:,}")

if duplicados != 0:
    raise ValueError(
        "Existen duplicados NIU-periodo."
    )

if negativos != 0:
    raise ValueError(
        "Hay consumos negativos. Revisar antes de modelar."
    )

In [ ]:
# ============================================================
# 4. CREAR MATRIZ DE CONSUMO NIU x MES
# ============================================================

periodo_min = (
    serie["periodo"]
    .min()
    .to_period("M")
    .to_timestamp()
)

periodo_max = (
    serie["periodo"]
    .max()
    .to_period("M")
    .to_timestamp()
)

meses = pd.date_range(
    start=periodo_min,
    end=periodo_max,
    freq="MS",
)

wide_consumo = (
    serie[
        [
            "NIU",
            "periodo",
            "consumo_kwh_mensual",
        ]
    ]
    .pivot(
        index="NIU",
        columns="periodo",
        values="consumo_kwh_mensual",
    )
    .reindex(
        columns=meses
    )
    .astype("float32")
)

nius = (
    wide_consumo.index
    .astype("string")
    .to_numpy()
)

matriz_consumo = wide_consumo.to_numpy(
    dtype="float32",
    copy=False,
)

mapa_mes = {
    pd.Timestamp(mes): i
    for i, mes in enumerate(
        wide_consumo.columns
    )
}

print("Matriz consumo:", matriz_consumo.shape)
print(
    "Memoria:",
    f"{matriz_consumo.nbytes / 1024**2:,.1f} MB"
)

del wide_consumo
gc.collect()

In [ ]:
# ============================================================
# 5. MATRIZ DE RÉGIMEN OBSERVADO / RECONSTRUIDO
# ============================================================
#
# 1 = reconstruido trimestral
# 0 = observado
# NaN = mes sin fila
#
# Esta variable NO define el perfil de consumo, pero sí entra
# como feature y permite auditar los errores por régimen.
# ============================================================

matriz_reconstruido = None

if (
    "origen_consumo" in serie.columns
    or "consumo_imputado" in serie.columns
):

    if "origen_consumo" in serie.columns:
        flag_origen = (
            serie["origen_consumo"]
            .astype("string")
            .str.contains(
                "reconstru",
                case=False,
                na=False,
            )
        )
    else:
        flag_origen = pd.Series(
            False,
            index=serie.index,
        )

    if "consumo_imputado" in serie.columns:
        flag_imputado = (
            serie["consumo_imputado"]
            .fillna(False)
            .astype(bool)
        )
    else:
        flag_imputado = pd.Series(
            False,
            index=serie.index,
        )

    serie["_es_reconstruido"] = (
        flag_origen
        | flag_imputado
    ).astype("float32")

    wide_regimen = (
        serie[
            [
                "NIU",
                "periodo",
                "_es_reconstruido",
            ]
        ]
        .pivot(
            index="NIU",
            columns="periodo",
            values="_es_reconstruido",
        )
        .reindex(
            index=nius,
            columns=meses,
        )
        .astype("float32")
    )

    matriz_reconstruido = (
        wide_regimen.to_numpy(
            dtype="float32",
            copy=False,
        )
    )

    print(
        "Matriz régimen creada:",
        matriz_reconstruido.shape
    )

    del wide_regimen
    gc.collect()

else:
    print(
        "No existen columnas de procedencia. "
        "El modelado continuará sin feature de régimen."
    )

In [ ]:
# ============================================================
# 6. FUNCIONES TEMPORALES
# ============================================================

def periodo_mes(fecha):
    return (
        pd.Timestamp(fecha)
        .to_period("M")
        .to_timestamp()
    )


def sumar_meses(
    fecha,
    delta,
):
    return (
        periodo_mes(fecha)
        .to_period("M")
        + delta
    ).to_timestamp()


def valores_mes(
    fecha,
    indices=None,
):
    fecha = periodo_mes(fecha)

    if indices is None:
        n = matriz_consumo.shape[0]
    else:
        n = len(indices)

    if fecha not in mapa_mes:
        return np.full(
            n,
            np.nan,
            dtype="float32",
        )

    columna = mapa_mes[fecha]

    if indices is None:
        return matriz_consumo[
            :,
            columna
        ]

    return matriz_consumo[
        indices,
        columna
    ]


def valores_regimen_mes(
    fecha,
    indices,
):
    if matriz_reconstruido is None:
        return np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    fecha = periodo_mes(fecha)

    if fecha not in mapa_mes:
        return np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    return matriz_reconstruido[
        indices,
        mapa_mes[fecha],
    ]


def target_horizonte(
    fecha_corte,
    horizonte,
    indices=None,
):
    fecha_target = sumar_meses(
        fecha_corte,
        horizonte,
    )

    return (
        valores_mes(
            fecha_target,
            indices,
        ),
        fecha_target,
    )

In [ ]:
# ============================================================
# 7. FUNCIONES DE VENTANA
# ============================================================

def ventana_consumo(
    fecha_corte,
    n_meses,
    indices,
):
    return np.column_stack(
        [
            valores_mes(
                sumar_meses(
                    fecha_corte,
                    -lag,
                ),
                indices,
            )
            for lag in range(n_meses)
        ]
    ).astype("float32")


def ventana_regimen(
    fecha_corte,
    n_meses,
    indices,
):
    return np.column_stack(
        [
            valores_regimen_mes(
                sumar_meses(
                    fecha_corte,
                    -lag,
                ),
                indices,
            )
            for lag in range(n_meses)
        ]
    ).astype("float32")


def media_nan(a):
    cuenta = np.sum(
        ~np.isnan(a),
        axis=1,
    )

    suma = np.nansum(
        a,
        axis=1,
    )

    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = cuenta > 0

    salida[mask] = (
        suma[mask]
        / cuenta[mask]
    )

    return salida


def mediana_nan(a):
    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = np.any(
        ~np.isnan(a),
        axis=1,
    )

    if mask.any():
        salida[mask] = np.nanmedian(
            a[mask],
            axis=1,
        )

    return salida


def std_nan(a):
    with np.errstate(
        invalid="ignore",
        divide="ignore",
    ):
        return np.nanstd(
            a,
            axis=1,
        ).astype("float32")


def min_nan(a):
    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = np.any(
        ~np.isnan(a),
        axis=1,
    )

    if mask.any():
        salida[mask] = np.nanmin(
            a[mask],
            axis=1,
        )

    return salida


def max_nan(a):
    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = np.any(
        ~np.isnan(a),
        axis=1,
    )

    if mask.any():
        salida[mask] = np.nanmax(
            a[mask],
            axis=1,
        )

    return salida

# Cálculo de perfiles de consumidor

La siguiente función es la pieza central del notebook.

Para cada NIU y fecha de corte calcula, usando únicamente información histórica:

- meses válidos;
- media 12m;
- mediana 12m;
- máximo 12m;
- desviación 12m;
- porcentaje de ceros;
- porcentaje de meses reconstruidos;
- perfil de consumidor.

La categoría `P3_GRANDE` se obtiene aquí mismo a partir de:

`mediana_12m >= UMBRAL_GRANDE_KWH`

Por defecto `UMBRAL_GRANDE_KWH = 5.000`.

La mediana evita clasificar como gran consumidor a un cliente que solo tuvo un pico aislado.

In [ ]:
# ============================================================
# 8. CALCULAR PERFIL DEL CONSUMIDOR EN UNA FECHA DE CORTE
# ============================================================

def calcular_perfiles(
    fecha_corte,
    indices=None,
):
    if indices is None:
        indices = np.arange(
            matriz_consumo.shape[0]
        )

    v12 = ventana_consumo(
        fecha_corte,
        12,
        indices,
    )

    validos12 = np.sum(
        ~np.isnan(v12),
        axis=1,
    )

    media12 = media_nan(v12)
    mediana12 = mediana_nan(v12)
    max12 = max_nan(v12)
    std12 = std_nan(v12)

    ceros12 = np.sum(
        np.where(
            np.isnan(v12),
            False,
            v12 == 0,
        ),
        axis=1,
    )

    pct_ceros12 = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    mask_validos = validos12 > 0

    pct_ceros12[mask_validos] = (
        ceros12[mask_validos]
        / validos12[mask_validos]
    )

    if matriz_reconstruido is not None:
        vr12 = ventana_regimen(
            fecha_corte,
            12,
            indices,
        )

        pct_reconstruido12 = media_nan(
            vr12
        )
    else:
        pct_reconstruido12 = np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    perfiles = np.full(
        len(indices),
        "P4_INSUFICIENTE",
        dtype=object,
    )

    historia_ok = (
        validos12
        >= MIN_MESES_VALIDOS_12
    )

    intermitente = (
        historia_ok
        & (
            (mediana12 <= UMBRAL_MUY_BAJO_KWH)
            | (
                pct_ceros12
                >= PCT_CEROS_INTERMITENTE
            )
        )
    )

    regular = (
        historia_ok
        & ~intermitente
        & (
            mediana12
            < UMBRAL_ALTO_KWH
        )
    )

    alto = (
        historia_ok
        & ~intermitente
        & (
            mediana12
            >= UMBRAL_ALTO_KWH
        )
        & (
            mediana12
            < UMBRAL_GRANDE_KWH
        )
    )

    grande = (
        historia_ok
        & (
            mediana12
            >= UMBRAL_GRANDE_KWH
        )
    )

    perfiles[
        intermitente
    ] = "P0_INTERMITENTE"

    perfiles[
        regular
    ] = "P1_REGULAR"

    perfiles[
        alto
    ] = "P2_ALTO"

    perfiles[
        grande
    ] = "P3_GRANDE"

    return pd.DataFrame(
        {
            "indice":
                indices,

            "NIU":
                nius[
                    indices
                ],

            "perfil":
                perfiles,

            "meses_validos_12m":
                validos12.astype(
                    "int8"
                ),

            "media_12m_kwh":
                media12,

            "mediana_12m_kwh":
                mediana12,

            "max_12m_kwh":
                max12,

            "std_12m_kwh":
                std12,

            "pct_ceros_12m":
                pct_ceros12,

            "pct_reconstruido_12m":
                pct_reconstruido12,
        }
    )

In [ ]:
# ============================================================
# 9. AUDITORÍA DE PERFILES EN EL ÚLTIMO MES DISPONIBLE
# ============================================================

FECHA_CORTE_FINAL = periodo_mes(
    periodo_max
)

perfiles_final = calcular_perfiles(
    FECHA_CORTE_FINAL
)

actual_final = valores_mes(
    FECHA_CORTE_FINAL
)

perfiles_final[
    "consumo_actual_kwh"
] = actual_final[
    perfiles_final["indice"]
]

resumen_perfiles_final = (
    perfiles_final
    .groupby(
        "perfil",
        as_index=False,
    )
    .agg(
        NIU=(
            "NIU",
            "nunique"
        ),
        consumo_actual_total_kwh=(
            "consumo_actual_kwh",
            "sum"
        ),
        mediana_historica_promedio_kwh=(
            "mediana_12m_kwh",
            "mean"
        ),
        media_historica_promedio_kwh=(
            "media_12m_kwh",
            "mean"
        ),
    )
)

total_energia = (
    resumen_perfiles_final[
        "consumo_actual_total_kwh"
    ].sum()
)

resumen_perfiles_final[
    "participacion_energia_pct"
] = np.where(
    total_energia > 0,
    (
        resumen_perfiles_final[
            "consumo_actual_total_kwh"
        ]
        / total_energia
        * 100
    ),
    np.nan,
)

display(
    resumen_perfiles_final
)

perfiles_final.to_parquet(
    RUTA_PERFILES_FINAL,
    index=False,
    engine="pyarrow",
)

In [ ]:
# ============================================================
# 10. CÁLCULO Y AUDITORÍA DE GRANDES CONSUMIDORES
# ============================================================
#
# Todo el cálculo de grandes consumidores queda dentro
# del notebook.
# ============================================================

grandes_final = (
    perfiles_final[
        perfiles_final["perfil"]
        .eq("P3_GRANDE")
    ]
    .copy()
    .sort_values(
        "media_12m_kwh",
        ascending=False,
    )
)

energia_grandes = (
    grandes_final[
        "consumo_actual_kwh"
    ].sum()
)

energia_total = (
    perfiles_final[
        "consumo_actual_kwh"
    ].sum()
)

participacion_grandes = (
    energia_grandes
    / energia_total
    * 100
    if energia_total > 0
    else np.nan
)

print("GRANDES CONSUMIDORES")
print("-" * 60)
print(
    f"Umbral mediana 12m : "
    f"{UMBRAL_GRANDE_KWH:,.0f} kWh"
)
print(
    f"NIU P3             : "
    f"{len(grandes_final):,}"
)
print(
    f"Energía último mes : "
    f"{energia_grandes:,.0f} kWh"
)
print(
    f"% energía total    : "
    f"{participacion_grandes:.2f}%"
)

print("\nDistribución histórica de P3:")

display(
    grandes_final[
        [
            "media_12m_kwh",
            "mediana_12m_kwh",
            "max_12m_kwh",
            "std_12m_kwh",
            "pct_reconstruido_12m",
        ]
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .T
)

print("\nTop 20 grandes consumidores:")

display(
    grandes_final[
        [
            "NIU",
            "consumo_actual_kwh",
            "media_12m_kwh",
            "mediana_12m_kwh",
            "max_12m_kwh",
            "pct_reconstruido_12m",
        ]
    ]
    .head(20)
)

grandes_final.to_parquet(
    RUTA_GRANDES_FINAL,
    index=False,
    engine="pyarrow",
)

In [ ]:
# ============================================================
# 11. SENSIBILIDAD DEL UMBRAL DE GRAN CONSUMIDOR
# ============================================================
#
# Esta tabla NO cambia automáticamente el umbral.
# Sirve para ver cómo cambia el tamaño y la participación
# energética de P3 con diferentes cortes.
# ============================================================

umbrales_grande = [
    2_000,
    3_000,
    5_000,
    7_500,
    10_000,
]

filas_umbrales = []

for umbral in umbrales_grande:

    mask = (
        perfiles_final[
            "meses_validos_12m"
        ]
        .ge(
            MIN_MESES_VALIDOS_12
        )
        & perfiles_final[
            "mediana_12m_kwh"
        ]
        .ge(
            umbral
        )
    )

    temp = perfiles_final[
        mask
    ]

    energia = (
        temp[
            "consumo_actual_kwh"
        ].sum()
    )

    filas_umbrales.append(
        {
            "umbral_mediana_12m_kwh":
                umbral,

            "NIU":
                temp[
                    "NIU"
                ].nunique(),

            "energia_actual_kwh":
                energia,

            "participacion_energia_pct":
                (
                    energia
                    / energia_total
                    * 100
                    if energia_total > 0
                    else np.nan
                ),
        }
    )

auditoria_umbrales = pd.DataFrame(
    filas_umbrales
)

display(
    auditoria_umbrales
)

auditoria_umbrales.to_csv(
    RUTA_AUDITORIA_UMBRALES,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
# ============================================================
# 12. GRÁFICA DE COMPOSICIÓN POR PERFIL
# ============================================================

plot_perfiles = (
    resumen_perfiles_final
    .sort_values(
        "participacion_energia_pct",
        ascending=False,
    )
)

ax = plot_perfiles.plot(
    x="perfil",
    y="participacion_energia_pct",
    kind="bar",
    figsize=(10, 5),
    legend=False,
)

ax.set_title(
    "Participación del consumo actual por perfil"
)

ax.set_xlabel(
    "Perfil"
)

ax.set_ylabel(
    "Participación de energía (%)"
)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Features de los modelos especializados

Cada modelo recibe únicamente información conocida hasta la fecha de corte:

- consumo actual;
- lags 1–12 y lag 24;
- mismo mes del año anterior;
- mismo mes de hace 2 años;
- medias, medianas y volatilidad;
- ceros recientes;
- crecimiento reciente;
- régimen observado/reconstruido;
- mes objetivo.

`NIU` nunca se utiliza como predictor.

In [ ]:
# ============================================================
# 13. CONSTRUIR FEATURES
# ============================================================

FEATURES = [
    "consumo_actual",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_4",
    "lag_5",
    "lag_6",
    "lag_12",
    "lag_24",
    "media_3m",
    "media_6m",
    "media_12m",
    "mediana_12m",
    "std_3m",
    "std_6m",
    "std_12m",
    "min_6m",
    "max_6m",
    "max_12m",
    "pct_ceros_6m",
    "pct_ceros_12m",
    "variacion_1m",
    "variacion_3m",
    "ratio_actual_media6",
    "mismo_mes_anio_anterior",
    "mismo_mes_2_anios",
    "reconstruido_actual",
    "pct_reconstruido_12m",
    "mes_objetivo",
    "mes_sin",
    "mes_cos",
]


def crear_features(
    fecha_corte,
    horizonte,
    indices,
):
    fecha_corte = periodo_mes(
        fecha_corte
    )

    fecha_objetivo = sumar_meses(
        fecha_corte,
        horizonte,
    )

    actual = valores_mes(
        fecha_corte,
        indices,
    )

    lags = {
        lag: valores_mes(
            sumar_meses(
                fecha_corte,
                -lag,
            ),
            indices,
        )
        for lag in [
            1, 2, 3, 4, 5, 6, 12, 24
        ]
    }

    v3 = ventana_consumo(
        fecha_corte,
        3,
        indices,
    )

    v6 = ventana_consumo(
        fecha_corte,
        6,
        indices,
    )

    v12 = ventana_consumo(
        fecha_corte,
        12,
        indices,
    )

    media6 = media_nan(v6)

    validos6 = np.sum(
        ~np.isnan(v6),
        axis=1,
    )

    validos12 = np.sum(
        ~np.isnan(v12),
        axis=1,
    )

    ceros6 = np.sum(
        np.where(
            np.isnan(v6),
            False,
            v6 == 0,
        ),
        axis=1,
    )

    ceros12 = np.sum(
        np.where(
            np.isnan(v12),
            False,
            v12 == 0,
        ),
        axis=1,
    )

    pct_ceros6 = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    pct_ceros12 = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    mask6 = validos6 > 0
    mask12 = validos12 > 0

    pct_ceros6[mask6] = (
        ceros6[mask6]
        / validos6[mask6]
    )

    pct_ceros12[mask12] = (
        ceros12[mask12]
        / validos12[mask12]
    )

    ratio = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    mask_ratio = (
        np.isfinite(actual)
        & np.isfinite(media6)
        & (media6 != 0)
    )

    ratio[mask_ratio] = (
        actual[mask_ratio]
        / media6[mask_ratio]
    )

    mismo_mes_1a = valores_mes(
        sumar_meses(
            fecha_objetivo,
            -12,
        ),
        indices,
    )

    mismo_mes_2a = valores_mes(
        sumar_meses(
            fecha_objetivo,
            -24,
        ),
        indices,
    )

    reconstruido_actual = (
        valores_regimen_mes(
            fecha_corte,
            indices,
        )
    )

    if matriz_reconstruido is not None:
        vr12 = ventana_regimen(
            fecha_corte,
            12,
            indices,
        )

        pct_reconstruido12 = (
            media_nan(vr12)
        )
    else:
        pct_reconstruido12 = np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    mes_obj = fecha_objetivo.month

    X = pd.DataFrame(
        {
            "consumo_actual":
                actual,

            "lag_1":
                lags[1],

            "lag_2":
                lags[2],

            "lag_3":
                lags[3],

            "lag_4":
                lags[4],

            "lag_5":
                lags[5],

            "lag_6":
                lags[6],

            "lag_12":
                lags[12],

            "lag_24":
                lags[24],

            "media_3m":
                media_nan(v3),

            "media_6m":
                media6,

            "media_12m":
                media_nan(v12),

            "mediana_12m":
                mediana_nan(v12),

            "std_3m":
                std_nan(v3),

            "std_6m":
                std_nan(v6),

            "std_12m":
                std_nan(v12),

            "min_6m":
                min_nan(v6),

            "max_6m":
                max_nan(v6),

            "max_12m":
                max_nan(v12),

            "pct_ceros_6m":
                pct_ceros6,

            "pct_ceros_12m":
                pct_ceros12,

            "variacion_1m":
                (
                    actual
                    - lags[1]
                ).astype("float32"),

            "variacion_3m":
                (
                    actual
                    - lags[3]
                ).astype("float32"),

            "ratio_actual_media6":
                ratio,

            "mismo_mes_anio_anterior":
                mismo_mes_1a,

            "mismo_mes_2_anios":
                mismo_mes_2a,

            "reconstruido_actual":
                reconstruido_actual,

            "pct_reconstruido_12m":
                pct_reconstruido12,

            "mes_objetivo":
                np.full(
                    len(indices),
                    mes_obj,
                    dtype="float32",
                ),

            "mes_sin":
                np.full(
                    len(indices),
                    np.sin(
                        2
                        * np.pi
                        * mes_obj
                        / 12
                    ),
                    dtype="float32",
                ),

            "mes_cos":
                np.full(
                    len(indices),
                    np.cos(
                        2
                        * np.pi
                        * mes_obj
                        / 12
                    ),
                    dtype="float32",
                ),
        }
    )

    return X[
        FEATURES
    ], fecha_objetivo

In [ ]:
# ============================================================
# 14. MÉTRICAS Y BASELINE
# ============================================================

def metricas_regresion(
    real,
    pred,
):
    real = np.asarray(
        real,
        dtype="float64",
    )

    pred = np.asarray(
        pred,
        dtype="float64",
    )

    mask = (
        np.isfinite(real)
        & np.isfinite(pred)
    )

    real = real[mask]
    pred = pred[mask]

    if len(real) == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "WAPE_pct": np.nan,
            "sMAPE_pct": np.nan,
            "R2": np.nan,
            "sesgo_pct": np.nan,
        }

    mae = mean_absolute_error(
        real,
        pred,
    )

    rmse = np.sqrt(
        mean_squared_error(
            real,
            pred,
        )
    )

    suma_real = np.abs(
        real
    ).sum()

    wape = (
        np.abs(
            real - pred
        ).sum()
        / suma_real
        * 100
        if suma_real > 0
        else np.nan
    )

    denom = (
        np.abs(real)
        + np.abs(pred)
    )

    mask_smape = denom > 0

    smape = (
        np.mean(
            2
            * np.abs(
                real[mask_smape]
                - pred[mask_smape]
            )
            / denom[mask_smape]
        )
        * 100
        if mask_smape.any()
        else 0.0
    )

    r2 = (
        r2_score(
            real,
            pred,
        )
        if len(real) > 1
        else np.nan
    )

    sesgo = (
        (
            pred.sum()
            - real.sum()
        )
        / real.sum()
        * 100
        if real.sum() != 0
        else np.nan
    )

    return {
        "n": int(len(real)),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "WAPE_pct": float(wape),
        "sMAPE_pct": float(smape),
        "R2": float(r2),
        "sesgo_pct": float(sesgo),
    }


def baseline_hibrido(
    fecha_corte,
    horizonte,
    indices,
):
    fecha_target = sumar_meses(
        fecha_corte,
        horizonte,
    )

    estacional = valores_mes(
        sumar_meses(
            fecha_target,
            -12,
        ),
        indices,
    )

    actual = valores_mes(
        fecha_corte,
        indices,
    )

    return np.where(
        np.isfinite(estacional),
        estacional,
        actual,
    ).astype("float32")

# Comparación de LightGBM, XGBoost y CatBoost

A partir de este punto los tres algoritmos reciben exactamente el mismo `X_train` y `y_train` para cada combinación de perfil y horizonte.

La competencia se realiza **solo en validación temporal**.

Para cada modelo se calculan dos resultados:

1. **ML puro**
2. **ML + baseline**, optimizando `alpha`

La selección final utiliza el menor `WAPE_blend_validacion_pct`. En caso de empate se usa como desempate el WAPE del ML puro.

In [ ]:
# ============================================================
# 15. FÁBRICAS DE MODELOS POR ALGORITMO
# ============================================================

def nuevo_clasificador(
    algoritmo,
):
    if algoritmo == "LightGBM":
        return LGBMClassifier(
            objective="binary",
            n_estimators=350,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_lambda=0.50,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )

    if algoritmo == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic",
            n_estimators=350,
            learning_rate=0.05,
            max_depth=8,
            min_child_weight=20,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_lambda=1.0,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if algoritmo == "CatBoost":
        return CatBoostClassifier(
            loss_function="Logloss",
            iterations=350,
            learning_rate=0.05,
            depth=8,
            l2_leaf_reg=5.0,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(
        f"Algoritmo no reconocido: {algoritmo}"
    )


def nuevo_regresor_log(
    algoritmo,
):
    if algoritmo == "LightGBM":
        return LGBMRegressor(
            objective="regression",
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=0.05,
            reg_lambda=0.50,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )

    if algoritmo == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror",
            n_estimators=500,
            learning_rate=0.04,
            max_depth=8,
            min_child_weight=20,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=0.05,
            reg_lambda=1.0,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if algoritmo == "CatBoost":
        return CatBoostRegressor(
            loss_function="RMSE",
            iterations=500,
            learning_rate=0.04,
            depth=8,
            l2_leaf_reg=5.0,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(
        f"Algoritmo no reconocido: {algoritmo}"
    )


def nuevo_regresor_tweedie(
    algoritmo,
    gran_consumidor=False,
):
    iteraciones = (
        650
        if gran_consumidor
        else 550
    )

    if algoritmo == "LightGBM":
        return LGBMRegressor(
            objective="tweedie",
            tweedie_variance_power=1.5,
            n_estimators=iteraciones,
            learning_rate=0.035,
            num_leaves=(
                31
                if gran_consumidor
                else 63
            ),
            min_child_samples=(
                20
                if gran_consumidor
                else 50
            ),
            subsample=0.95,
            colsample_bytree=0.95,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )

    if algoritmo == "XGBoost":
        return XGBRegressor(
            objective="reg:tweedie",
            tweedie_variance_power=1.5,
            n_estimators=iteraciones,
            learning_rate=0.035,
            max_depth=(
                6
                if gran_consumidor
                else 8
            ),
            min_child_weight=(
                10
                if gran_consumidor
                else 20
            ),
            subsample=0.95,
            colsample_bytree=0.95,
            reg_alpha=0.05,
            reg_lambda=1.0,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if algoritmo == "CatBoost":
        return CatBoostRegressor(
            loss_function="Tweedie:variance_power=1.5",
            iterations=iteraciones,
            learning_rate=0.035,
            depth=(
                7
                if gran_consumidor
                else 8
            ),
            l2_leaf_reg=5.0,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(
        f"Algoritmo no reconocido: {algoritmo}"
    )

In [ ]:
# ============================================================
# 16. CONSTRUIR TRAIN PARA UN PERFIL Y HORIZONTE
# ============================================================

def construir_train_perfil_h(
    perfil,
    horizonte,
    max_target_train,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    max_target_train = periodo_mes(
        max_target_train
    )

    ultimo_origen = sumar_meses(
        max_target_train,
        -horizonte,
    )

    origenes = pd.date_range(
        start=PRIMER_ORIGEN_TRAIN,
        end=ultimo_origen,
        freq="MS",
    )

    X_partes = []
    y_partes = []
    auditoria = []

    limite = (
        MAX_MUESTRA_POR_ORIGEN[
            perfil
        ]
    )

    for numero, origen in enumerate(
        origenes,
        start=1,
    ):
        perfiles_origen = calcular_perfiles(
            origen
        )

        idx_perfil = (
            perfiles_origen.loc[
                perfiles_origen[
                    "perfil"
                ].eq(perfil),
                "indice",
            ]
            .to_numpy(
                dtype="int64"
            )
        )

        if len(idx_perfil) == 0:
            continue

        y_all, fecha_target = (
            target_horizonte(
                origen,
                horizonte,
                idx_perfil,
            )
        )

        candidatos = idx_perfil[
            np.isfinite(
                y_all
            )
        ]

        if len(candidatos) == 0:
            continue

        if (
            limite is not None
            and len(candidatos) > limite
        ):
            seleccion = rng.choice(
                candidatos,
                size=limite,
                replace=False,
            )
        else:
            seleccion = candidatos

        y_sel, _ = target_horizonte(
            origen,
            horizonte,
            seleccion,
        )

        X_sel, _ = crear_features(
            origen,
            horizonte,
            seleccion,
        )

        X_partes.append(
            X_sel
        )

        y_partes.append(
            y_sel.astype(
                "float32"
            )
        )

        auditoria.append(
            {
                "perfil":
                    perfil,

                "horizonte":
                    horizonte,

                "origen":
                    origen,

                "target":
                    fecha_target,

                "candidatos":
                    len(candidatos),

                "muestra":
                    len(seleccion),
            }
        )

        if (
            numero == 1
            or numero % 6 == 0
            or numero == len(origenes)
        ):
            print(
                f"{perfil} | h={horizonte} | "
                f"{numero}/{len(origenes)} | "
                f"origen={origen:%Y-%m} | "
                f"muestra={len(seleccion):,}"
            )

        del perfiles_origen
        del X_sel
        gc.collect()

    if not X_partes:
        return (
            pd.DataFrame(
                columns=FEATURES
            ),
            np.array(
                [],
                dtype="float32",
            ),
            pd.DataFrame(),
        )

    X = pd.concat(
        X_partes,
        ignore_index=True,
    )

    y = np.concatenate(
        y_partes
    ).astype("float32")

    auditoria = pd.DataFrame(
        auditoria
    )

    del X_partes
    del y_partes
    gc.collect()

    return X, y, auditoria

In [ ]:
# ============================================================
# 17. ENTRENAR / PREDECIR UN ALGORITMO SEGÚN EL PERFIL
# ============================================================

def entrenar_modelo_algoritmo(
    perfil,
    algoritmo,
    X,
    y,
):
    if len(y) == 0:
        return None

    # --------------------------------------------------------
    # P0: hurdle = clasificador + regresor positivo
    # --------------------------------------------------------

    if perfil == "P0_INTERMITENTE":

        y_bin = (
            y > 0
        ).astype("int8")

        bundle = {
            "perfil":
                perfil,

            "algoritmo":
                algoritmo,

            "tipo":
                "hurdle",

            "prob_constante":
                float(
                    y_bin.mean()
                ),

            "classifier":
                None,

            "regressor":
                None,
        }

        if np.unique(
            y_bin
        ).size >= 2:

            clf = nuevo_clasificador(
                algoritmo
            )

            clf.fit(
                X,
                y_bin,
            )

            bundle[
                "classifier"
            ] = clf

        positivos = (
            y > 0
        )

        if positivos.any():

            reg = nuevo_regresor_log(
                algoritmo
            )

            reg.fit(
                X.loc[
                    positivos
                ],
                np.log1p(
                    y[
                        positivos
                    ]
                ),
            )

            bundle[
                "regressor"
            ] = reg

        return bundle

    # --------------------------------------------------------
    # P1: log1p
    # --------------------------------------------------------

    if perfil == "P1_REGULAR":

        reg = nuevo_regresor_log(
            algoritmo
        )

        reg.fit(
            X,
            np.log1p(
                np.maximum(
                    y,
                    0,
                )
            ),
        )

        return {
            "perfil":
                perfil,

            "algoritmo":
                algoritmo,

            "tipo":
                "log",

            "regressor":
                reg,
        }

    # --------------------------------------------------------
    # P2/P3: Tweedie en kWh
    # --------------------------------------------------------

    if perfil in [
        "P2_ALTO",
        "P3_GRANDE",
    ]:

        reg = nuevo_regresor_tweedie(
            algoritmo=algoritmo,
            gran_consumidor=(
                perfil
                == "P3_GRANDE"
            ),
        )

        reg.fit(
            X,
            np.maximum(
                y,
                0,
            ),
        )

        return {
            "perfil":
                perfil,

            "algoritmo":
                algoritmo,

            "tipo":
                "tweedie",

            "regressor":
                reg,
        }

    raise ValueError(
        f"Perfil no reconocido: {perfil}"
    )


def predecir_bundle(
    bundle,
    X,
):
    if bundle is None:
        return np.full(
            len(X),
            np.nan,
            dtype="float32",
        )

    tipo = bundle[
        "tipo"
    ]

    algoritmo = bundle[
        "algoritmo"
    ]

    if tipo == "hurdle":

        if (
            bundle[
                "classifier"
            ]
            is None
        ):
            p_pos = np.full(
                len(X),
                bundle[
                    "prob_constante"
                ],
                dtype="float32",
            )
        else:
            p_pos = (
                bundle[
                    "classifier"
                ]
                .predict_proba(
                    X
                )[:, 1]
                .astype(
                    "float32"
                )
            )

        if (
            bundle[
                "regressor"
            ]
            is None
        ):
            consumo_positivo = np.zeros(
                len(X),
                dtype="float32",
            )
        else:
            consumo_positivo = np.maximum(
                np.expm1(
                    bundle[
                        "regressor"
                    ].predict(
                        X
                    )
                ),
                0,
            ).astype(
                "float32"
            )

        return (
            p_pos
            * consumo_positivo
        ).astype(
            "float32"
        )

    if tipo == "log":

        return np.maximum(
            np.expm1(
                bundle[
                    "regressor"
                ].predict(
                    X
                )
            ),
            0,
        ).astype(
            "float32"
        )

    if tipo == "tweedie":

        if algoritmo == "CatBoost":

            pred = (
                bundle[
                    "regressor"
                ]
                .predict(
                    X,
                    prediction_type="Exponent",
                )
            )

        else:

            pred = (
                bundle[
                    "regressor"
                ]
                .predict(
                    X
                )
            )

        return np.maximum(
            pred,
            0,
        ).astype(
            "float32"
        )

    raise ValueError(
        f"Tipo de bundle no reconocido: {tipo}"
    )

In [ ]:
# ============================================================
# 18. COMPONENTES DE MÉTRICAS ACUMULABLES
# ============================================================

def componentes_vacios():
    return {
        "n":
            0,

        "abs_error_sum":
            0.0,

        "sq_error_sum":
            0.0,

        "abs_real_sum":
            0.0,

        "real_sum":
            0.0,

        "real_sq_sum":
            0.0,

        "pred_sum":
            0.0,

        "smape_sum":
            0.0,

        "smape_n":
            0,
    }


def actualizar_componentes(
    comp,
    real,
    pred,
):
    real = np.asarray(
        real,
        dtype="float64",
    )

    pred = np.asarray(
        pred,
        dtype="float64",
    )

    mask = (
        np.isfinite(real)
        & np.isfinite(pred)
    )

    real = real[mask]
    pred = pred[mask]

    if len(real) == 0:
        return comp

    error = (
        pred
        - real
    )

    comp[
        "n"
    ] += len(
        real
    )

    comp[
        "abs_error_sum"
    ] += float(
        np.abs(
            error
        ).sum()
    )

    comp[
        "sq_error_sum"
    ] += float(
        np.square(
            error
        ).sum()
    )

    comp[
        "abs_real_sum"
    ] += float(
        np.abs(
            real
        ).sum()
    )

    comp[
        "real_sum"
    ] += float(
        real.sum()
    )

    comp[
        "real_sq_sum"
    ] += float(
        np.square(
            real
        ).sum()
    )

    comp[
        "pred_sum"
    ] += float(
        pred.sum()
    )

    denom = (
        np.abs(
            real
        )
        + np.abs(
            pred
        )
    )

    mask_smape = (
        denom > 0
    )

    if mask_smape.any():

        comp[
            "smape_sum"
        ] += float(
            (
                2
                * np.abs(
                    real[
                        mask_smape
                    ]
                    - pred[
                        mask_smape
                    ]
                )
                / denom[
                    mask_smape
                ]
            ).sum()
        )

        comp[
            "smape_n"
        ] += int(
            mask_smape.sum()
        )

    return comp


def metricas_componentes(
    comp,
):
    n = comp[
        "n"
    ]

    if n == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "WAPE_pct": np.nan,
            "sMAPE_pct": np.nan,
            "R2": np.nan,
            "sesgo_pct": np.nan,
        }

    mae = (
        comp[
            "abs_error_sum"
        ]
        / n
    )

    rmse = np.sqrt(
        comp[
            "sq_error_sum"
        ]
        / n
    )

    wape = (
        comp[
            "abs_error_sum"
        ]
        / comp[
            "abs_real_sum"
        ]
        * 100
        if comp[
            "abs_real_sum"
        ] > 0
        else np.nan
    )

    smape = (
        comp[
            "smape_sum"
        ]
        / comp[
            "smape_n"
        ]
        * 100
        if comp[
            "smape_n"
        ] > 0
        else np.nan
    )

    media_real = (
        comp[
            "real_sum"
        ]
        / n
    )

    sst = (
        comp[
            "real_sq_sum"
        ]
        - n
        * media_real**2
    )

    r2 = (
        1
        - (
            comp[
                "sq_error_sum"
            ]
            / sst
        )
        if sst > 0
        else np.nan
    )

    sesgo = (
        (
            comp[
                "pred_sum"
            ]
            - comp[
                "real_sum"
            ]
        )
        / comp[
            "real_sum"
        ]
        * 100
        if comp[
            "real_sum"
        ] != 0
        else np.nan
    )

    return {
        "n":
            int(
                n
            ),

        "MAE":
            float(
                mae
            ),

        "RMSE":
            float(
                rmse
            ),

        "WAPE_pct":
            float(
                wape
            ),

        "sMAPE_pct":
            float(
                smape
            ),

        "R2":
            float(
                r2
            ),

        "sesgo_pct":
            float(
                sesgo
            ),
    }

In [ ]:
# ============================================================
# 19. OBTENER DATASET DE VALIDACIÓN PARA UN GRUPO
# ============================================================

def obtener_validacion_grupo(
    perfil,
    horizonte,
    fecha_corte,
):
    perfiles_corte = calcular_perfiles(
        fecha_corte
    )

    indices = (
        perfiles_corte.loc[
            perfiles_corte[
                "perfil"
            ].eq(
                perfil
            ),
            "indice",
        ]
        .to_numpy(
            dtype="int64"
        )
    )

    if len(indices) == 0:
        return None

    y_real, fecha_target = (
        target_horizonte(
            fecha_corte,
            horizonte,
            indices,
        )
    )

    mask = np.isfinite(
        y_real
    )

    indices = indices[
        mask
    ]

    y_real = y_real[
        mask
    ].astype(
        "float32"
    )

    if len(indices) == 0:
        return None

    X, _ = crear_features(
        fecha_corte,
        horizonte,
        indices,
    )

    baseline = baseline_hibrido(
        fecha_corte,
        horizonte,
        indices,
    )

    return {
        "indices":
            indices,

        "X":
            X,

        "real":
            y_real,

        "baseline":
            baseline,

        "fecha_target":
            fecha_target,
    }

## Optuna y utilidades de la búsqueda

In [ ]:
# ============================================================
# 22. LIBRERÍAS DE OPTIMIZACIÓN
# ============================================================

import json as _json  # nombre distinto para no chocar con nada del notebook

try:
    import optuna
    from optuna.trial import FixedTrial
except ImportError as e:
    raise ImportError(
        "Falta Optuna. Ejecuta: %pip install -U optuna"
    ) from e

optuna.logging.set_verbosity(optuna.logging.WARNING)

RUTA_OPTUNA_DB = SALIDA_DIR / "optuna_estudios.sqlite3"
RUTA_HIPERPARAMETROS = SALIDA_DIR / "hiperparametros_optimos_por_grupo.json"
RUTA_SELECCION_OPTIMIZADA = SALIDA_DIR / "seleccion_modelo_por_perfil_horizonte_optimizada.csv"
RUTA_MODELOS_OPTIMIZADOS = SALIDA_DIR / "modelos_ganadores_optimizados.joblib"

print("Base de datos Optuna     :", RUTA_OPTUNA_DB)
print("Hiperparámetros óptimos  :", RUTA_HIPERPARAMETROS)
print("Selección optimizada     :", RUTA_SELECCION_OPTIMIZADA)
print("Bundles optimizados      :", RUTA_MODELOS_OPTIMIZADOS)


## Cargar los modelos ganadores ya elegidos en la comparación

In [ ]:
# ============================================================
# 23. CARGAR LA SELECCIÓN DE MODELOS GANADORES
# ============================================================
# Este archivo lo genera el notebook de comparación
# (Modelado_segmentado_comparacion_modelos_3_6_meses), celda
# "21. TABLAS DE COMPARACIÓN Y SELECCIÓN". No se re-corre aquí la
# competencia LightGBM vs XGBoost vs CatBoost: solo se afina el
# algoritmo que ya ganó en cada perfil x horizonte.
# ============================================================

if not RUTA_SELECCION_MODELOS.exists():
    raise FileNotFoundError(
        f"No existe el archivo:\n{RUTA_SELECCION_MODELOS}\n"
        "Corre primero el notebook de comparación de modelos."
    )

seleccion_modelos_base = pd.read_csv(RUTA_SELECCION_MODELOS)

seleccion_modelos_base = seleccion_modelos_base[
    seleccion_modelos_base["perfil"].isin(PERFILES_MODELADOS)
].reset_index(drop=True)

print(f"Grupos perfil x horizonte a optimizar: {len(seleccion_modelos_base)}")
display(
    seleccion_modelos_base
    .sort_values(["perfil", "horizonte"])
)


## Espacios de búsqueda y fábricas de modelos parametrizadas

In [ ]:
# ============================================================
# 24. ESPACIO DE BÚSQUEDA POR ALGORITMO
# ============================================================
# Los rangos están centrados alrededor de los valores fijos que
# ya usaba nuevo_clasificador / nuevo_regresor_log / nuevo_regresor_tweedie
# en el notebook de comparación, para no alejarse de configuraciones
# que ya se sabe que funcionan razonablemente bien.
#
# El prefijo evita colisiones de nombre cuando un mismo trial ajusta
# más de un submodelo en la misma llamada (caso P0: clasificador + regresor).
# ============================================================

def sugerir_hparams_arbol(trial, prefijo, algoritmo, es_tweedie=False):

    def p(nombre):
        return f"{prefijo}_{nombre}"

    if algoritmo == "LightGBM":
        params = {
            "n_estimators": trial.suggest_int(p("n_estimators"), 200, 1000, step=50),
            "learning_rate": trial.suggest_float(p("learning_rate"), 0.01, 0.15, log=True),
            "num_leaves": trial.suggest_int(p("num_leaves"), 15, 255, log=True),
            "min_child_samples": trial.suggest_int(p("min_child_samples"), 10, 200, log=True),
            "subsample": trial.suggest_float(p("subsample"), 0.6, 1.0),
            "colsample_bytree": trial.suggest_float(p("colsample_bytree"), 0.6, 1.0),
            "reg_alpha": trial.suggest_float(p("reg_alpha"), 1e-3, 5.0, log=True),
            "reg_lambda": trial.suggest_float(p("reg_lambda"), 1e-3, 5.0, log=True),
        }
        if es_tweedie:
            params["tweedie_variance_power"] = trial.suggest_float(p("tweedie_variance_power"), 1.1, 1.9)
        return params

    if algoritmo == "XGBoost":
        params = {
            "n_estimators": trial.suggest_int(p("n_estimators"), 200, 1000, step=50),
            "learning_rate": trial.suggest_float(p("learning_rate"), 0.01, 0.15, log=True),
            "max_depth": trial.suggest_int(p("max_depth"), 3, 12),
            "min_child_weight": trial.suggest_float(p("min_child_weight"), 1.0, 50.0, log=True),
            "subsample": trial.suggest_float(p("subsample"), 0.6, 1.0),
            "colsample_bytree": trial.suggest_float(p("colsample_bytree"), 0.6, 1.0),
            "reg_alpha": trial.suggest_float(p("reg_alpha"), 1e-3, 5.0, log=True),
            "reg_lambda": trial.suggest_float(p("reg_lambda"), 1e-3, 5.0, log=True),
        }
        if es_tweedie:
            params["tweedie_variance_power"] = trial.suggest_float(p("tweedie_variance_power"), 1.1, 1.9)
        return params

    if algoritmo == "CatBoost":
        params = {
            "iterations": trial.suggest_int(p("iterations"), 200, 1000, step=50),
            "learning_rate": trial.suggest_float(p("learning_rate"), 0.01, 0.15, log=True),
            "depth": trial.suggest_int(p("depth"), 4, 10),
            "l2_leaf_reg": trial.suggest_float(p("l2_leaf_reg"), 1.0, 10.0, log=True),
        }
        if es_tweedie:
            params["tweedie_variance_power"] = trial.suggest_float(p("tweedie_variance_power"), 1.1, 1.9)
        return params

    raise ValueError(f"Algoritmo no reconocido: {algoritmo}")


def construir_clasificador_optuna(trial, algoritmo, prefijo="clf"):
    hp = sugerir_hparams_arbol(trial, prefijo, algoritmo)

    if algoritmo == "LightGBM":
        return LGBMClassifier(
            objective="binary", random_state=SEED, n_jobs=-1, verbosity=-1, **hp,
        )
    if algoritmo == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic", tree_method="hist",
            random_state=SEED, n_jobs=-1, verbosity=0, **hp,
        )
    if algoritmo == "CatBoost":
        return CatBoostClassifier(
            loss_function="Logloss", random_seed=SEED, verbose=False,
            allow_writing_files=False, thread_count=-1, **hp,
        )
    raise ValueError(f"Algoritmo no reconocido: {algoritmo}")


def construir_regresor_log_optuna(trial, algoritmo, prefijo="reg"):
    hp = sugerir_hparams_arbol(trial, prefijo, algoritmo)

    if algoritmo == "LightGBM":
        return LGBMRegressor(
            objective="regression", random_state=SEED, n_jobs=-1, verbosity=-1, **hp,
        )
    if algoritmo == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror", tree_method="hist",
            random_state=SEED, n_jobs=-1, verbosity=0, **hp,
        )
    if algoritmo == "CatBoost":
        return CatBoostRegressor(
            loss_function="RMSE", random_seed=SEED, verbose=False,
            allow_writing_files=False, thread_count=-1, **hp,
        )
    raise ValueError(f"Algoritmo no reconocido: {algoritmo}")


def construir_regresor_tweedie_optuna(trial, algoritmo, prefijo="reg"):
    hp = sugerir_hparams_arbol(trial, prefijo, algoritmo, es_tweedie=True)
    potencia = hp.pop("tweedie_variance_power")

    if algoritmo == "LightGBM":
        return LGBMRegressor(
            objective="tweedie", tweedie_variance_power=potencia,
            random_state=SEED, n_jobs=-1, verbosity=-1, **hp,
        )
    if algoritmo == "XGBoost":
        return XGBRegressor(
            objective="reg:tweedie", tweedie_variance_power=potencia,
            tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0, **hp,
        )
    if algoritmo == "CatBoost":
        return CatBoostRegressor(
            loss_function=f"Tweedie:variance_power={potencia:.3f}",
            random_seed=SEED, verbose=False,
            allow_writing_files=False, thread_count=-1, **hp,
        )
    raise ValueError(f"Algoritmo no reconocido: {algoritmo}")


In [ ]:
# ============================================================
# 25. ENTRENAR UN BUNDLE CON LOS HIPERPARÁMETROS DE UN TRIAL
# ============================================================
# Misma lógica de entrenar_modelo_algoritmo (hurdle / log / tweedie
# según el perfil), pero usando las fábricas parametrizadas por Optuna.
# Funciona tanto con un trial "vivo" (durante la búsqueda) como con un
# optuna.trial.FixedTrial (al reentrenar con los mejores hiperparámetros).
# ============================================================

def entrenar_modelo_algoritmo_optuna(perfil, algoritmo, X, y, trial):
    if len(y) == 0:
        return None

    if perfil == "P0_INTERMITENTE":

        y_bin = (y > 0).astype("int8")

        bundle = {
            "perfil": perfil,
            "algoritmo": algoritmo,
            "tipo": "hurdle",
            "prob_constante": float(y_bin.mean()),
            "classifier": None,
            "regressor": None,
        }

        if np.unique(y_bin).size >= 2:
            clf = construir_clasificador_optuna(trial, algoritmo, prefijo="clf")
            clf.fit(X, y_bin)
            bundle["classifier"] = clf

        positivos = y > 0

        if positivos.any():
            reg = construir_regresor_log_optuna(trial, algoritmo, prefijo="reg")
            reg.fit(X.loc[positivos], np.log1p(y[positivos]))
            bundle["regressor"] = reg

        return bundle

    if perfil == "P1_REGULAR":
        reg = construir_regresor_log_optuna(trial, algoritmo, prefijo="reg")
        reg.fit(X, np.log1p(np.maximum(y, 0)))

        return {
            "perfil": perfil, "algoritmo": algoritmo,
            "tipo": "log", "regressor": reg,
        }

    if perfil in ["P2_ALTO", "P3_GRANDE"]:
        reg = construir_regresor_tweedie_optuna(trial, algoritmo, prefijo="reg")
        reg.fit(X, np.maximum(y, 0))

        return {
            "perfil": perfil, "algoritmo": algoritmo,
            "tipo": "tweedie", "regressor": reg,
        }

    raise ValueError(f"Perfil no reconocido: {perfil}")


## Función objetivo: WAPE del blend óptimo en validación

In [ ]:
# ============================================================
# 26. FUNCIÓN OBJETIVO DE OPTUNA
# ============================================================
# Entrena el bundle con los hiperparámetros del trial, lo evalúa en
# AMBOS cortes de validación (2024-07-01 y 2025-01-01, agregados,
# igual que en la comparación original), vuelve a buscar el mejor
# alpha con la misma GRID_ALPHA, y devuelve el WAPE del blend óptimo.
# El backtest (2025-07-01) NO se usa aquí.
# ============================================================

def objetivo_optuna(trial, perfil, horizonte, algoritmo, X_train, y_train):

    bundle = entrenar_modelo_algoritmo_optuna(
        perfil, algoritmo, X_train, y_train, trial
    )

    comp_ml = componentes_vacios()

    comp_alpha = {
        float(a): componentes_vacios()
        for a in GRID_ALPHA
    }

    for corte in CORTES_VALIDACION:

        val = obtener_validacion_grupo(
            perfil=perfil, horizonte=horizonte, fecha_corte=corte,
        )

        if val is None:
            continue

        pred_ml = predecir_bundle(bundle, val["X"])
        real = val["real"]
        baseline = val["baseline"]

        actualizar_componentes(comp_ml, real, pred_ml)

        for alpha in GRID_ALPHA:
            alpha = float(alpha)
            pred_blend = alpha * pred_ml + (1 - alpha) * baseline
            actualizar_componentes(comp_alpha[alpha], real, pred_blend)

        del val, pred_ml
        gc.collect()

    candidatos_alpha = []

    for alpha, comp in comp_alpha.items():
        met = metricas_componentes(comp)
        if np.isfinite(met["WAPE_pct"]):
            candidatos_alpha.append((met["WAPE_pct"], alpha))

    del bundle
    gc.collect()

    if not candidatos_alpha:
        return float("inf")

    candidatos_alpha.sort(key=lambda x: (x[0], -x[1]))
    mejor_wape_blend, mejor_alpha = candidatos_alpha[0]

    trial.set_user_attr("alpha_ml", mejor_alpha)
    trial.set_user_attr("WAPE_ML_pct", metricas_componentes(comp_ml)["WAPE_pct"])

    return mejor_wape_blend


## Bucle principal de optimización

Ajusta `N_TRIALS_POR_GRUPO` y, si quieres un tope de tiempo por grupo,
`TIMEOUT_POR_GRUPO_MIN` antes de correr esta celda. Como pediste población
completa y granularidad por perfil × horizonte, cada uno de los 24 grupos
(P0-P3 × 6 horizontes) entrena con TODOS los clientes candidatos de ese
perfil — igual que hacía la comparación original — así que cada intento de
Optuna no es gratis. Con los estudios guardados en SQLite puedes interrumpir
esta celda en cualquier momento y volver a correrla más tarde: retoma los
intentos ya hechos por grupo en vez de empezar de cero.

Sugerencia práctica: empieza con `N_TRIALS_POR_GRUPO` bajo (15-20) para tener
una primera pasada completa por los 24 grupos y ver dónde vale la pena
invertir más intentos, en vez de gastar horas en un solo grupo antes de ver
los demás.

In [ ]:
# ============================================================
# 27. BUCLE PRINCIPAL: UN ESTUDIO DE OPTUNA POR PERFIL x HORIZONTE
# ============================================================

N_TRIALS_POR_GRUPO = 25
TIMEOUT_POR_GRUPO_MIN = None  # p.ej. 90 para limitar a 90 min por grupo

resultados_optuna = []
mejores_hparams = {}

for fila in seleccion_modelos_base.itertuples(index=False):

    perfil = fila.perfil
    horizonte = int(fila.horizonte)
    algoritmo = fila.modelo_ganador
    wape_blend_original = fila.WAPE_blend_validacion_pct

    clave = f"{perfil}_h{horizonte}_{algoritmo}"

    print("\n" + "=" * 80)
    print(f"OPTUNA | {clave} | WAPE blend actual = {wape_blend_original:.2f}%")
    print("=" * 80)

    inicio_grupo = time.time()

    X_train, y_train, _audit = construir_train_perfil_h(
        perfil=perfil,
        horizonte=horizonte,
        max_target_train=MAX_TARGET_TRAIN_COMPARACION,
        seed=SEED + horizonte,
    )

    print("Train:", X_train.shape, y_train.shape)

    if len(y_train) == 0:
        print("Sin datos suficientes. Se omite este grupo.")
        continue

    # Semilla en el sampler: sin ella, dos corridas de este notebook exploran
    # hiperparámetros distintos y pueden elegir modelos distintos. Con ella,
    # la búsqueda es reproducible (misma serie -> mismos hiperparámetros).
    estudio = optuna.create_study(
        study_name=clave,
        direction="minimize",
        storage=f"sqlite:///{RUTA_OPTUNA_DB}",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=SEED + horizonte),
    )

    ya_corridos = len(estudio.trials)

    if ya_corridos >= N_TRIALS_POR_GRUPO:
        print(f"Ya tiene {ya_corridos} intentos guardados >= objetivo. Se omite.")
    else:
        estudio.optimize(
            lambda trial: objetivo_optuna(
                trial, perfil, horizonte, algoritmo, X_train, y_train
            ),
            n_trials=N_TRIALS_POR_GRUPO - ya_corridos,
            timeout=(TIMEOUT_POR_GRUPO_MIN * 60 if TIMEOUT_POR_GRUPO_MIN else None),
            gc_after_trial=True,
        )

    mejor_trial = estudio.best_trial

    resultados_optuna.append({
        "perfil": perfil,
        "horizonte": horizonte,
        "algoritmo": algoritmo,
        "WAPE_blend_original_pct": wape_blend_original,
        "WAPE_blend_optimizado_pct": mejor_trial.value,
        "mejora_pct_puntos": wape_blend_original - mejor_trial.value,
        "alpha_ml_optimizado": mejor_trial.user_attrs.get("alpha_ml"),
        "n_trials": len(estudio.trials),
    })

    mejores_hparams[clave] = mejor_trial.params

    print(
        f"Mejor WAPE blend: {mejor_trial.value:.2f}% "
        f"(antes: {wape_blend_original:.2f}%) | "
        f"alpha={mejor_trial.user_attrs.get('alpha_ml')} | "
        f"tiempo grupo={(time.time() - inicio_grupo) / 60:.1f} min"
    )

    del X_train, y_train
    gc.collect()

resultados_optuna = pd.DataFrame(resultados_optuna)

display(
    resultados_optuna
    .sort_values(["perfil", "horizonte"])
)

resultados_optuna.to_csv(
    RUTA_SELECCION_OPTIMIZADA, index=False, encoding="utf-8-sig",
)

with open(RUTA_HIPERPARAMETROS, "w", encoding="utf-8") as f:
    _json.dump(mejores_hparams, f, indent=2, ensure_ascii=False)

print(f"\nGuardado: {RUTA_SELECCION_OPTIMIZADA}")
print(f"Guardado: {RUTA_HIPERPARAMETROS}")


## Reentrenar los bundles ganadores con los mejores hiperparámetros

In [ ]:
# ============================================================
# 28. REENTRENAR CON LOS MEJORES HIPERPARÁMETROS Y GUARDAR
# ============================================================
# optuna.trial.FixedTrial deja reutilizar las mismas fábricas
# parametrizadas pasando un diccionario de hiperparámetros ya fijos,
# en vez de un trial "vivo" que sigue explorando.
# ============================================================

bundles_optimizados = {}

for fila in resultados_optuna.itertuples(index=False):

    perfil = fila.perfil
    horizonte = int(fila.horizonte)
    algoritmo = fila.algoritmo
    clave = f"{perfil}_h{horizonte}_{algoritmo}"

    print(f"Reentrenando {clave} con los mejores hiperparámetros...")

    X_train, y_train, _audit = construir_train_perfil_h(
        perfil=perfil,
        horizonte=horizonte,
        max_target_train=MAX_TARGET_TRAIN_COMPARACION,
        seed=SEED + horizonte,
    )

    trial_fijo = FixedTrial(mejores_hparams[clave])

    bundle = entrenar_modelo_algoritmo_optuna(
        perfil, algoritmo, X_train, y_train, trial_fijo,
    )

    bundles_optimizados[clave] = bundle

    del X_train, y_train
    gc.collect()

joblib.dump(bundles_optimizados, RUTA_MODELOS_OPTIMIZADOS)

print(f"\nGuardado: {RUTA_MODELOS_OPTIMIZADOS}")
print(
    "\nSiguiente paso natural: correr estos bundles contra el backtest "
    "(2025-07-01) igual que el notebook de comparación, para confirmar que "
    "la mejora de validación se sostiene en el juez independiente."
)
